In [ ]:
# [CELL 1]: Cài đặt Hugging Face Transformers, PyTorch và thư viện tách từ tiếng Việt
!pip install -q transformers torch pyvi

import torch
from transformers import AutoTokenizer, AutoModel
from pyvi import ViTokenizer

# Kiểm tra xem GPU đã sẵn sàng chưa
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"-> Môi trường tính toán đang sử dụng: {device}")

-> Môi trường tính toán đang sử dụng: cpu


In [ ]:
# [CELL 2]: Tải PhoBERT-base-v2 chính thức của VinAI Research
MODEL_NAME = "vinai/phobert-base-v2"

print(f"Đang tải mô hình {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()  # Chuyển sang chế độ trích xuất đặc trưng (evaluation)

print("-> [XONG Ô 2]: Đã tải và nạp xong PhoBERT vào GPU!")

Đang tải mô hình vinai/phobert-base-v2...


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


-> [XONG Ô 2]: Đã tải và nạp xong PhoBERT vào GPU!


In [ ]:
# [CELL 3]: Chạy kiểm thử mã hóa câu tiếng Việt với PhoBERT
raw_sentence = "Học sinh sinh viên nghiên cứu xử lý ngôn ngữ tự nhiên."

print(f"1. Câu văn gốc ban đầu:\n   '{raw_sentence}'\n")

# BƯỚC 1: Tiền xử lý tách từ ghép tiếng Việt (Bắt buộc với PhoBERT)
segmented_sentence = ViTokenizer.tokenize(raw_sentence)
print(f"2. Sau khi chạy qua bộ tách từ ghép (ViTokenizer):\n   '{segmented_sentence}'\n")

# BƯỚC 2: Tokenize thành các Subwords theo từ điển 64.000 từ của PhoBERT
tokens = tokenizer.tokenize(segmented_sentence)
print(f"3. Danh sách các token/subword PhoBERT nhận diện:")
print(f"   {tokens}\n")

# BƯỚC 3: Chuyển thành Tensor IDs và đưa qua mạng Transformer Encoder
inputs = tokenizer(segmented_sentence, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    # Lấy ma trận vector của tất cả các từ trong câu (Hidden States)
    last_hidden_state = outputs.last_hidden_state
    # Lấy vector đặc trưng ngữ cảnh đại diện cho toàn bộ câu (tại vị trí token [CLS] đầu tiên)
    cls_vector = last_hidden_state[:, 0, :].squeeze().cpu().numpy()

print(f"4. Kích thước ma trận ngữ cảnh của cả câu (Batch, Số token, Số chiều):")
print(f"   {last_hidden_state.shape}")
print(f"5. Kích thước vector đại diện ngữ cảnh toàn câu ([CLS]):")
print(f"   {cls_vector.shape} chiều (Chuẩn RoBERTa-base: 768 chiều)")
print(f"6. 5 giá trị số thực đầu tiên của vector [CLS]:")
print(f"   {cls_vector[:5]}")

1. Câu văn gốc ban đầu:
   'Học sinh sinh viên nghiên cứu xử lý ngôn ngữ tự nhiên.'

2. Sau khi chạy qua bộ tách từ ghép (ViTokenizer):
   'Học_sinh sinh_viên nghiên_cứu xử_lý ngôn_ngữ tự_nhiên .'

3. Danh sách các token/subword PhoBERT nhận diện:
   ['Học_sinh', 'sinh_viên', 'nghiên_cứu', 'xử_lý', 'ngôn_ngữ', 'tự_nhiên', '.']

4. Kích thước ma trận ngữ cảnh của cả câu (Batch, Số token, Số chiều):
   torch.Size([1, 9, 768])
5. Kích thước vector đại diện ngữ cảnh toàn câu ([CLS]):
   (768,) chiều (Chuẩn RoBERTa-base: 768 chiều)
6. 5 giá trị số thực đầu tiên của vector [CLS]:
   [ 0.04448332  0.3233326  -0.29570735 -0.69664973 -0.25511196]


In [ ]:
# [CELL 1 - viBERT]: Nạp mô hình viBERT-base-cased chính thức từ FPT AI
from transformers import AutoTokenizer, AutoModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Đường dẫn mô hình viBERT chuẩn trên Hugging Face
VIBERT_MODEL_NAME = "FPTAI/vibert-base-cased"

print(f"Đang tải mô hình {VIBERT_MODEL_NAME}...")
vibert_tokenizer = AutoTokenizer.from_pretrained(VIBERT_MODEL_NAME)
vibert_model = AutoModel.from_pretrained(VIBERT_MODEL_NAME).to(device)
vibert_model.eval()

print("-> [XONG BƯỚC 1]: Đã tải thành công viBERT vào bộ nhớ GPU!")

Đang tải mô hình FPTAI/vibert-base-cased...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/255k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  581MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  581MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: FPTAI/vibert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> [XONG BƯỚC 1]: Đã tải thành công viBERT vào bộ nhớ GPU!


In [ ]:
# [CELL 2 - viBERT]: Kiểm nghiệm băm Subword và trích xuất đặc trưng
test_sentence = "Học sinh sinh viên nghiên cứu xử lý ngôn ngữ tự nhiên."

print(f"1. Câu văn gốc đưa vào (Không cần tách từ trước):\n   '{test_sentence}'\n")

# BƯỚC 1: WordPiece tự động băm nhỏ câu thành các tokens / subwords
tokens = vibert_tokenizer.tokenize(test_sentence)
print(f"2. Danh sách tokens do WordPiece của viBERT phân tách:")
print(f"   {tokens}\n")

# BƯỚC 2: Chuyển tokens thành Tensor IDs đưa vào mạng Transformer
inputs = vibert_tokenizer(test_sentence, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = vibert_model(**inputs)
    # Ma trận biểu diễn toàn bộ các token trong câu
    last_hidden_state = outputs.last_hidden_state
    # Vector đặc trưng ngữ cảnh đại diện toàn câu (tại token [CLS] đầu tiên)
    cls_vector = last_hidden_state[:, 0, :].squeeze().cpu().numpy()

print(f"3. Kích thước ma trận ngữ cảnh đầu ra (Batch, Tokens, Dimensions):")
print(f"   {last_hidden_state.shape}")
print(f"4. Kích thước vector đại diện ngữ cảnh toàn câu ([CLS]):")
print(f"   {cls_vector.shape} chiều (Chuẩn BERT-base: 768 chiều)")
print(f"5. 5 phần tử số thực đầu tiên của vector [CLS]:")
print(f"   {cls_vector[:5]}")

1. Câu văn gốc đưa vào (Không cần tách từ trước):
   'Học sinh sinh viên nghiên cứu xử lý ngôn ngữ tự nhiên.'

2. Danh sách tokens do WordPiece của viBERT phân tách:
   ['hoc', 'sinh', 'sinh', 'vien', 'nghi', '##en', 'cu', '##u', 'x', '##u', 'ly', 'ng', '##on', 'ng', '##u', 'tu', 'n', '##hien', '.']

3. Kích thước ma trận ngữ cảnh đầu ra (Batch, Tokens, Dimensions):
   torch.Size([1, 21, 768])
4. Kích thước vector đại diện ngữ cảnh toàn câu ([CLS]):
   (768,) chiều (Chuẩn BERT-base: 768 chiều)
5. 5 phần tử số thực đầu tiên của vector [CLS]:
   [ 0.02030415  0.23482946  0.22887063 -0.75995404  0.25660875]


In [ ]:
# [CELL 1 - XLM-RoBERTa]: Tải mô hình xlm-roberta-base chính thức
from transformers import AutoTokenizer, AutoModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tên mô hình chính thức trên Hugging Face
XLM_MODEL_NAME = "xlm-roberta-base"

print(f"Đang tải {XLM_MODEL_NAME} (khoảng 1.1GB, mất tầm 30-45 giây)...")
xlm_tokenizer = AutoTokenizer.from_pretrained(XLM_MODEL_NAME)
xlm_model = AutoModel.from_pretrained(XLM_MODEL_NAME).to(device)
xlm_model.eval()

print("-> [XONG BƯỚC 1]: Đã tải thành công XLM-RoBERTa vào GPU!")

Đang tải xlm-roberta-base (khoảng 1.1GB, mất tầm 30-45 giây)...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> [XONG BƯỚC 1]: Đã tải thành công XLM-RoBERTa vào GPU!


In [ ]:
# [CELL 2 - XLM-RoBERTa]: Kiểm nghiệm SentencePiece trên câu pha trộn Anh - Việt (Code-mixing)
test_sentence = "Áo này order về mặc bao cool ngầu, feedback 5 sao nha shop!"

print(f"1. Câu kiểm thử (Code-mixing thực tế trên E-commerce):\n   '{test_sentence}'\n")

# BƯỚC 1: Phân tách Subwords bằng SentencePiece (Từ điển 250k)
tokens = xlm_tokenizer.tokenize(test_sentence)
print(f"2. Danh sách Subwords do SentencePiece băm ra:")
print(f"   {tokens}\n")

# BƯỚC 2: Chuyển chuỗi thành Tensor đưa vào mạng Transformer
inputs = xlm_tokenizer(test_sentence, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = xlm_model(**inputs)
    last_hidden_state = outputs.last_hidden_state
    # Trích xuất vector biểu diễn ngữ cảnh toàn câu tại token đầu tiên [CLS]
    cls_vector = last_hidden_state[:, 0, :].squeeze().cpu().numpy()

print(f"3. Kích thước ma trận ngữ cảnh đầu ra (Batch, Tokens, Dimensions):")
print(f"   {last_hidden_state.shape}")
print(f"4. Kích thước vector đặc trưng toàn câu ([CLS]):")
print(f"   {cls_vector.shape} chiều (Chuẩn RoBERTa: 768 chiều)")
print(f"5. 5 giá trị số thực đầu tiên của vector [CLS]:")
print(f"   {cls_vector[:5]}")

1. Câu kiểm thử (Code-mixing thực tế trên E-commerce):
   'Áo này order về mặc bao cool ngầu, feedback 5 sao nha shop!'

2. Danh sách Subwords do SentencePiece băm ra:
   ['▁Á', 'o', '▁này', '▁order', '▁về', '▁mặc', '▁bao', '▁cool', '▁ng', 'ầ', 'u', ',', '▁feedback', '▁5', '▁sao', '▁nha', '▁shop', '!']

3. Kích thước ma trận ngữ cảnh đầu ra (Batch, Tokens, Dimensions):
   torch.Size([1, 20, 768])
4. Kích thước vector đặc trưng toàn câu ([CLS]):
   (768,) chiều (Chuẩn RoBERTa: 768 chiều)
5. 5 giá trị số thực đầu tiên của vector [CLS]:
   [ 0.07837726  0.05659289  0.09238685 -0.00107578 -0.0213664 ]
